# Data Understanding and Cleaning

In [8]:
# Step 1: Environment & Package Setup
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display configurations
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported successfully.")

# Step 2: Load Dataset from Excel
RAW_DATA_PATH = r"C:\Users\AADHYA SHARMA\Downloads\customer-churn-revenue-intelligence\data\raw\ecommerce_churn.csv"
PROCESSED_DATA_PATH = r"C:\Users\AADHYA SHARMA\Downloads\customer-churn-revenue-intelligence\data\processed\customer_churn_cleaned.csv"

# Load the customer data sheet
try:
    df_raw = pd.read_excel(RAW_DATA_PATH, sheet_name='E Comm')
except Exception:
    # If sheet name is default
    df_raw = pd.read_excel(RAW_DATA_PATH)

print(f"Raw Dataset Loaded Successfully. Shape: {df_raw.shape[0]} rows, {df_raw.shape} columns")
df_raw.head()

# Step 3: Data Understanding & Inspection
print("\n--- Summary Info ---")
df_raw.info()

print("\n--- Missing Values Audit ---")
missing_df = pd.DataFrame({
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Percentage': (df_raw.isnull().sum() / len(df_raw)) * 100
})
print(missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False))

print("\n--- Duplicate Records Audit ---")
duplicate_count = df_raw.duplicated(subset=['CustomerID']).sum()
print(f"Duplicate Customer IDs Found: {duplicate_count}")

# Step 4: Categorical Inconsistency Audit
categorical_cols = df_raw.select_dtypes(include=['object']).columns.tolist()
print("\n--- Unique Values in Categorical Columns ---")
for col in categorical_cols:
    print(f"{col}: {df_raw[col].unique().tolist()}")

# Step 5: Data Cleaning Implementation
df_clean = df_raw.copy()

# 5.1 Standardize Inconsistent Categorical Strings
category_replacements = {
    'PreferredLoginDevice': {'Phone': 'Mobile Phone'},
    'PreferredPaymentMode': {
        'COD': 'Cash on Delivery',
        'CC': 'Credit Card'
    },
    'PreferedOrderCat': {
        'Mobile': 'Mobile Phone'
    }
}

for col, mapping in category_replacements.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace(mapping)
        print(f"Standardized column: {col} -> Current classes: {df_clean[col].unique().tolist()}")

# 5.2 Numeric Missing Value Imputation (Using Median to preserve distribution without skew)
numeric_impute_cols = [
    'Tenure', 
    'WarehouseToHome', 
    'HourSpendOnApp', 
    'OrderAmountHikeFromlastYear', 
    'CouponUsed', 
    'OrderCount', 
    'DaySinceLastOrder'
]

for col in numeric_impute_cols:
    if col in df_clean.columns and df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"Imputed missing values in '{col}' with median = {median_val:.2f}")

# 5.3 Type Casting & Integrity Checks
int_cols = ['CustomerID', 'Churn', 'CityTier', 'NumberOfDeviceRegistered', 'SatisfactionScore', 
            'NumberOfAddress', 'Complain', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder']

for col in int_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(int)

# 5.4 Outlier Validation & Handling
# Verify WarehouseToHome distance bounds (reasonable cap at 100 km for domestic e-commerce delivery)
outliers_distance = (df_clean['WarehouseToHome'] > 100).sum()
print(f"Records with WarehouseToHome > 100km: {outliers_distance}")
df_clean.loc[df_clean['WarehouseToHome'] > 100, 'WarehouseToHome'] = 100.0

# Step 6: Post-Cleaning Validation Audit
print("\n" + "="*50)
print("DATA CLEANING AUDIT REPORT")
print("="*50)
print(f"Initial Rows: {len(df_raw)} | Cleaned Rows: {len(df_clean)}")
print(f"Initial Columns: {df_raw.shape[1]} | Cleaned Columns: {df_clean.shape[1]}")
print(f"Total Remaining Missing Values: {df_clean.isnull().sum().sum()}")
print(f"Unique Customers: {df_clean['CustomerID'].nunique()}")
print(f"Overall Churn Count: {df_clean['Churn'].sum()} ({(df_clean['Churn'].mean()*100):.2f}%)")

# Step 7: Export Cleaned Dataset
import os

# Automatically create the processed directory if it doesn't exist
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)

try:
    df_clean.to_csv(PROCESSED_DATA_PATH, index=False)
    print(f"Cleaned dataset successfully saved to:\n{PROCESSED_DATA_PATH}")
except PermissionError:
    # If the file is locked by Excel, save with a fallback timestamped filename
    fallback_path = PROCESSED_DATA_PATH.replace(".csv", "_v1.csv")
    df_clean.to_csv(fallback_path, index=False)
    print(f"Warning: Original file was locked. Cleaned dataset saved to:\n{fallback_path}")

Libraries imported successfully.
Raw Dataset Loaded Successfully. Shape: 5630 rows, (5630, 20) columns

--- Summary Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  Satisfact

In [9]:
import os
import shutil

BASE_DIR = r"C:\Users\AADHYA SHARMA\Downloads\customer-churn-revenue-intelligence"
PROCESSED_FOLDER = os.path.join(BASE_DIR, "data", "processed")
CLEAN_FILE_PATH = os.path.join(PROCESSED_FOLDER, "ecommerce_churn_clean.csv")

# 1. Ensure the directory exists
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

# 2. Check if a directory with the file name was accidentally created and remove it
target_path = os.path.join(PROCESSED_FOLDER, "customer_churn_cleaned.csv")
if os.path.exists(target_path) and os.path.isdir(target_path):
    print(f"Removing accidental directory: {target_path}")
    shutil.rmtree(target_path)

# 3. Export clean dataset to CSV
df_clean.to_csv(CLEAN_FILE_PATH, index=False)
print(f"Cleaned dataset successfully saved as a file to:\n{CLEAN_FILE_PATH}")

Removing accidental directory: C:\Users\AADHYA SHARMA\Downloads\customer-churn-revenue-intelligence\data\processed\customer_churn_cleaned.csv
Cleaned dataset successfully saved as a file to:
C:\Users\AADHYA SHARMA\Downloads\customer-churn-revenue-intelligence\data\processed\ecommerce_churn_clean.csv
